In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
# 使用tavily作为web搜索工具

from langchain_core.runnables import RunnableConfig

from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


load_dotenv(find_dotenv())

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")



In [2]:
model = init_chat_model(
    model="qwen3.5-35b-a3b",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    # 尝试显式关闭思考模式
    extra_body={"enable_thinking": False} 
)

In [3]:
def get_weather(city: str) -> str:
    """获取给定城市的天气。"""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model= model,
    tools=[get_weather]
)

for msg in agent.stream(
    {"messages":[{"role":"user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in msg.items():
        print(f"step:{step}")
        print(f"contentL:{data['messages'][-1].content_blocks}")

step:model
contentL:[{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_ace684e6402f4c40980a8040'}]
step:tools
contentL:[{'type': 'text', 'text': "It's always sunny in SF!"}]
step:model
contentL:[{'type': 'text', 'text': 'The weather in SF is sunny!'}]


In [4]:
def get_weather(city: str) -> str:
    """获取给定城市的天气。"""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model= model,
    tools=[get_weather]
)

for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode = "messages",
):
    print(f"node: {metadata['langgraph_node']}")
    print(f"content: {token.content_blocks}")
    print("\n")

node: model
content: [{'type': 'tool_call_chunk', 'id': 'call_d9e6e8a24aca4780b6b5855f', 'name': 'get_weather', 'args': '', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '', 'name': None, 'args': '{"city": "SF', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '', 'name': None, 'args': '', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '', 'name': None, 'args': '"', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '', 'name': None, 'args': '}', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '', 'name': None, 'args': '', 'index': 0}]


node: model
content: []


node: model
content: []


node: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]


node: model
content: [{'type': 'text', 'text': 'The'}]


node: model
content: [{'type': 'text', 'text': ' weather in SF'}]


node: model
content: [{'type': 'text', 'text': ' is sunny'}]


node: model
content: [{'

In [5]:
for stream_mode, chunk in agent.stream(  # [!code highlight]
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    print(f"content: {chunk}")
    print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 276, 'total_tokens': 301, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 25}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 276}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-35b-a3b', 'system_fingerprint': None, 'id': 'chatcmpl-61499c3e-b71e-9385-96c7-3b30ffbc6d69', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d52d5-a01b-7912-9229-a1c48e98d5fb-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_a0d7bcf5189e4bad97959f87', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 25, 'total_tokens': 301, 'input_token_details': {}, 'output_token_details':